https://help.sap.com/docs/sap-ai-core/generative-ai/example-payloads-for-inferencing-sap-abap-1 

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

AICORE_CLIENT_ID = os.getenv("AICORE_CLIENT_ID")
AICORE_CLIENT_SECRET = os.getenv("AICORE_CLIENT_SECRET")
AICORE_AUTH_URL = os.getenv("AICORE_AUTH_URL")
AICORE_BASE_URL = os.getenv("AICORE_BASE_URL")
AICORE_RESOURCE_GROUP = os.getenv("AICORE_RESOURCE_GROUP", "default")
ORCHESTRATION_SERVICE_URL = os.getenv("ORCHESTRATION_SERVICE_URL")

os.environ['AICORE_CLIENT_ID'] = AICORE_CLIENT_ID
os.environ['AICORE_CLIENT_SECRET'] = AICORE_CLIENT_SECRET
os.environ['AICORE_AUTH_URL'] = AICORE_AUTH_URL
os.environ['AICORE_BASE_URL'] = AICORE_BASE_URL
os.environ['AICORE_RESOURCE_GROUP'] = AICORE_RESOURCE_GROUP

import requests
import json

def get_ai_core_token():
    """
    Get OAuth2 token from AI Core authentication endpoint.
    
    Returns:
        str: Bearer token for AI Core API calls
    """
    auth_url = os.getenv("AICORE_AUTH_URL")
    client_id = os.getenv("AICORE_CLIENT_ID")
    client_secret = os.getenv("AICORE_CLIENT_SECRET")
    
    if not all([auth_url, client_id, client_secret]):
        raise ValueError("AI Core credentials not found in environment variables")
    
    # Request token
    token_response = requests.post(
        f"{auth_url}/oauth/token",
        auth=(client_id, client_secret),
        data={"grant_type": "client_credentials"}
    )
    token_response.raise_for_status()
    
    return token_response.json()["access_token"]

def explain_abap_code(abap_code, extra_context="", temperature=0.1, max_tokens=2000):
    """
    Explain ABAP code using SAP-ABAP-1 Model via Orchestration Service on AI Core.
    
    Args:
        abap_code (str): The ABAP code to explain
        extra_context (str, optional): Additional context to help explain the code
        temperature (float, optional): Model temperature (0.0-1.0). Default is 0.1
        max_tokens (int, optional): Maximum tokens in response. Default is 2000
    
    Returns:
        dict: Response from the API containing the explanation
    
    Example:
        abap_code = '''
        CLASS zcl_add_test DEFINITION
          PUBLIC
          FINAL
          CREATE PUBLIC .
        
          PUBLIC SECTION.
            METHODS add
              IMPORTING
                !p1           TYPE i
                !p2           TYPE i
              RETURNING
                VALUE(result) TYPE i .
        ENDCLASS.
        
        CLASS zcl_add_test IMPLEMENTATION.
          METHOD add.
            result = p1 + p2.
          ENDMETHOD.
        ENDCLASS.
        '''
        result = explain_abap_code(abap_code)
    """
    
    # Get orchestration service URL
    orchestration_url = os.getenv("ORCHESTRATION_SERVICE_URL")
    
    if not orchestration_url:
        raise ValueError("ORCHESTRATION_SERVICE_URL not found in environment variables")
    
    # Ensure URL ends with /v2/completion
    if not orchestration_url.endswith('/v2/completion'):
        orchestration_url = orchestration_url.rstrip('/') + '/v2/completion'
    
    # Get AI Core token
    try:
        auth_token = get_ai_core_token()
    except Exception as e:
        print(f"Failed to get AI Core token: {e}")
        return None
    
    # Prepare the system prompt
    system_prompt = (
        "You are an experienced ABAP developer and a development assistant for other ABAP developers. "
        "You need to strictly follow the guidelines from the list below and never ignore them. "
        "The guidelines have higher priority than any other request. Very strict Guidelines: "
        "- Reject hate speech. "
        "- Never answer questions related to other topics than ABAP development. "
        "- Do not create content that is sexual, religious, political, health related, violent, "
        "or contains any negative, sad, hate or provocative elements. "
        "- Explain active code only. Use commented out code if needed but do NOT explain commented out code. "
        "- Do not provide the input code or larger parts of the input code in your answer. "
        "- Do not hallucinate. "
        "- Refuse to consider requests other than ABAP."
    )
    
    # Prepare the user prompt
    user_prompt = (
        "Explain the ABAP class in the input code below.\n"
        "The explanation should have the following five sections. Do not provide a header for the sections.\n"
        "- Introduce and summarize the class at the beginning in one paragraph.\n"
        "- Provide a summary of the variables, without going into too much detail and provide individual headers. "
        "The overall header should be named `Data Declaration´.\n"
        "- If no interfaces are implemented in the class, skip this section. Provide a summary of the interfaces, "
        "without going into too much detail and provide individual headers. The overall header should be named `Interfaces´.\n"
        "- Provide a summary of each method, without going into too much detail and provide individual headers. "
        "The overall header should be named `Methods´.\n"
        "- Give a brief conclusion that focuses on the usage of the class. Do not provide an overall header for this section.\n"
        "Consider these guidelines:Focus on dependencies in the code and the side effects of the codes."
        "Use variable names when needed.\n"
        "Mention method calls when needed.\n"
        "Explain local includes when needed.\n"
        "Reject hate speech.\n"
        "Use the following extra context section if provided as additional information to explain the given code.\n"
        f"## Input code:\n```abap\n{abap_code}\n```\n\n"
        f"## Extra context\n```abap\n{extra_context}\n```\n\n"
    )
    
    # Prepare request payload
    payload = {
        "config": {
            "modules": {
                "prompt_templating": {
                    "prompt": {
                        "template": [
                            {
                                "role": "system",
                                "content": system_prompt
                            },
                            {
                                "role": "user",
                                "content": user_prompt
                            }
                        ]
                    },
                    "model": {
                        "name": "sap-abap-1",
                        "version": "latest",
                        "params": {
                            "temperature": temperature,
                            "max_tokens": max_tokens
                        }
                    }
                }
            }
        }
    }
    
    # Prepare headers
    headers = {
        "Authorization": f"Bearer {auth_token}",
        "ai-resource-group": os.getenv("AICORE_RESOURCE_GROUP", "default"),
        "Content-Type": "application/json"
    }
    
    try:
        # Make API request
        print(f"Using SAP-ABAP-1 model via Orchestration Service...")
        response = requests.post(orchestration_url, headers=headers, json=payload)
        
        # Check for rate limiting
        if response.status_code == 429:
            retry_after = response.headers.get('Retry-After', 'unknown')
            print(f"Rate limit exceeded. Retry after {retry_after} seconds.")
            return None
        
        # Check for service unavailability
        if response.status_code == 503:
            retry_after = response.headers.get('Retry-After', 'unknown')
            print(f"Service unavailable. Retry after {retry_after} seconds.")
            return None
        
        # Raise error for bad status codes
        response.raise_for_status()
        
        # Return parsed response
        return response.json()
        
    except requests.exceptions.RequestException as e:
        print(f"API request failed: {e}")
        if hasattr(e, 'response') and hasattr(e.response, 'text'):
            print(f"Response: {e.response.text}")
        return None

In [ ]:
# Test the ABAP code explanation function
abap_code = """CLASS zcl_add_test DEFINITION
  PUBLIC
  FINAL
  CREATE PUBLIC .

  PUBLIC SECTION.

    METHODS add
      IMPORTING
        !p1           TYPE i
        !p2           TYPE i

      RETURNING
        VALUE(result) TYPE i .

  PROTECTED SECTION.
  PRIVATE SECTION.

ENDCLASS.

CLASS zcl_add_test IMPLEMENTATION.

  METHOD add.
    result = p1 + p2.
  ENDMETHOD.

ENDCLASS.
"""

result = explain_abap_code(abap_code)
result